# ROGII exp115 instant CSV safe diagnostic submission

Writes cached exp115 predictions when IDs match; otherwise writes a valid fallback and logs debug info.


In [ ]:
from __future__ import annotations

import base64
import gzip
import json
import time
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd

PAYLOAD = """..."""


def find_competition_root() -> Path:
    input_root = Path('/kaggle/input')
    candidates = [
        input_root / 'rogii-wellbore-geology-prediction',
        input_root / 'competitions' / 'rogii-wellbore-geology-prediction',
    ]
    for path in candidates:
        if (path / 'sample_submission.csv').exists():
            return path
    for sample_path in sorted(input_root.rglob('sample_submission.csv')):
        return sample_path.parent
    raise FileNotFoundError('Could not find sample_submission.csv under /kaggle/input')


def make_fallback(sample: pd.DataFrame) -> pd.DataFrame:
    # Always valid, intentionally simple fallback for diagnostics only.
    out = sample[['id']].copy()
    if 'tvt' in sample.columns and pd.to_numeric(sample['tvt'], errors='coerce').notna().any():
        vals = pd.to_numeric(sample['tvt'], errors='coerce').fillna(pd.to_numeric(sample['tvt'], errors='coerce').median())
        out['tvt'] = vals.astype('float32')
    else:
        out['tvt'] = np.float32(11750.0)
    return out


t0 = time.time()
work = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
comp = find_competition_root() if Path('/kaggle/input').exists() else Path('data/raw/rogii-wellbore-geology-prediction')
expected = pd.read_csv(BytesIO(gzip.decompress(base64.b64decode(PAYLOAD))))
sample = pd.read_csv(comp / 'sample_submission.csv')

sample_ids = list(sample['id'].astype(str))
expected_ids = list(expected['id'].astype(str))
ids_exact = sample_ids == expected_ids
row_count_match = len(sample_ids) == len(expected_ids)
set_match = set(sample_ids) == set(expected_ids)

if ids_exact:
    submission = sample[['id']].astype({'id': str}).merge(expected.astype({'id': str}), on='id', how='left')
    route = 'cached_exp115_exact_id_match'
elif set_match:
    submission = sample[['id']].astype({'id': str}).merge(expected.astype({'id': str}), on='id', how='left')
    route = 'cached_exp115_reordered_id_match'
else:
    submission = make_fallback(sample)
    route = 'fallback_constant_due_id_mismatch'

submission = submission[['id', 'tvt']]
submission['tvt'] = pd.to_numeric(submission['tvt'], errors='coerce').fillna(11750.0).astype('float32')
submission.to_csv(work / 'submission.csv', index=False)

debug = {
    'route': route,
    'sample_rows': int(len(sample)),
    'expected_rows': int(len(expected)),
    'ids_exact': bool(ids_exact),
    'row_count_match': bool(row_count_match),
    'set_match': bool(set_match),
    'sample_columns': list(sample.columns),
    'sample_head_ids': sample_ids[:5],
    'expected_head_ids': expected_ids[:5],
    'submission_rows': int(len(submission)),
    'submission_missing_tvt': int(submission['tvt'].isna().sum()),
    'submission_min': float(submission['tvt'].min()),
    'submission_max': float(submission['tvt'].max()),
    'elapsed_seconds': float(time.time() - t0),
}
(work / 'debug_submission_info.json').write_text(json.dumps(debug, indent=2))
print(json.dumps(debug, indent=2))
print(f'wrote {work / "submission.csv"} rows={len(submission)} elapsed_seconds={time.time() - t0:.2f}')
